# 模型剪枝教程 (Model Pruning Tutorial)

> **前置知识**: PyTorch 基础、深度学习模型训练流程
>
> **学习目标**: 掌握剪枝的原理、类型和实现方法

---

## 为什么需要剪枝？

```
┌─────────────────────────────────────────────────────────────┐
│                    剪枝的核心价值                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  过参数化假说:                                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  神经网络通常是过参数化的                           │   │
│  │  大量参数是冗余的，对输出贡献很小                   │   │
│  │  移除这些参数不会显著影响模型性能                   │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  彩票假说 (Lottery Ticket Hypothesis):                      │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  在随机初始化的网络中，存在一个稀疏子网络           │   │
│  │  这个子网络从头训练可以达到原始网络的性能           │   │
│  │  → 说明大部分参数确实是冗余的                       │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  剪枝效果:                                                  │
│  - 模型大小减少 5-10x                                      │
│  - 计算量减少 2-4x                                         │
│  - 精度损失 <1%                                            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **剪枝基础** - 理解剪枝的原理和类型
2. **非结构化剪枝** - 移除单个权重
3. **结构化剪枝** - 移除整个通道/神经元
4. **重要性评估** - 不同的重要性指标
5. **迭代剪枝** - 逐步剪枝并微调

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
import matplotlib.pyplot as plt
import numpy as np

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

print("=" * 50)
print("环境准备完成")
print("=" * 50)
print(f"PyTorch 版本: {torch.__version__}")

## 1. 剪枝基础

**核心概念**: 剪枝通过移除模型中不重要的参数来减小模型大小和计算量

```
┌─────────────────────────────────────────────────────────────┐
│                     剪枝类型对比                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  非结构化剪枝 (Unstructured Pruning)                        │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  原始矩阵          剪枝后 (稀疏矩阵)                │   │
│  │  [1.2  0.3  0.8]   [1.2  0    0.8]                 │   │
│  │  [0.1  2.1  0.4] → [0    2.1  0  ]                 │   │
│  │  [0.9  0.2  1.5]   [0.9  0    1.5]                 │   │
│  │                                                     │   │
│  │  特点:                                              │   │
│  │  - 移除单个权重 (置零)                             │   │
│  │  - 产生稀疏矩阵                                    │   │
│  │  - 压缩率高 (可达 90%+)                            │   │
│  │  - 需要特殊硬件/软件支持才能加速                   │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  结构化剪枝 (Structured Pruning)                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  原始矩阵          剪枝后 (小矩阵)                  │   │
│  │  [1.2  0.3  0.8]   [1.2  0.8]                      │   │
│  │  [0.1  2.1  0.4] → [0.1  0.4]                      │   │
│  │  [0.9  0.2  1.5]   [0.9  1.5]                      │   │
│  │                                                     │   │
│  │  特点:                                              │   │
│  │  - 移除整个结构 (通道、层、注意力头)               │   │
│  │  - 产生规则的小模型                                │   │
│  │  - 无需特殊硬件支持                                │   │
│  │  - 直接获得加速效果                                │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 定义测试模型
# ============================================================

class ConvNet(nn.Module):
    """
    简单卷积神经网络
    
    结构: Conv → Pool → Conv → Pool → FC → FC
    用于演示剪枝效果
    """
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)   # 1→32 通道
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)  # 32→64 通道
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 28→14
        x = self.pool(F.relu(self.conv2(x)))  # 14→7
        x = x.view(x.size(0), -1)             # 展平
        x = F.relu(self.fc1(x))
        return self.fc2(x)


# 创建模型
model = ConvNet()

# 统计参数
def count_parameters(model):
    """统计模型参数量"""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_parameters(model)

print("=" * 60)
print("测试模型信息")
print("=" * 60)
print(f"\n模型结构:")
print(f"  Conv1: 1→32 通道, 3x3 卷积")
print(f"  Conv2: 32→64 通道, 3x3 卷积")
print(f"  FC1: 3136→128")
print(f"  FC2: 128→10")
print(f"\n总参数量: {total:,}")
print(f"可训练参数: {trainable:,}")

In [ ]:
# ============================================================
# 可视化权重分布
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

layers = [
    ('conv1', model.conv1.weight),
    ('conv2', model.conv2.weight),
    ('fc1', model.fc1.weight),
    ('fc2', model.fc2.weight)
]

for ax, (name, weight) in zip(axes.flatten(), layers):
    w = weight.detach().flatten().numpy()
    ax.hist(w, bins=50, alpha=0.7, color='blue', edgecolor='black')
    ax.axvline(x=0, color='red', linestyle='--', label='零点')
    ax.set_title(f'{name} 权重分布 (参数量: {weight.numel():,})', fontsize=11)
    ax.set_xlabel('权重值')
    ax.set_ylabel('频数')
    ax.legend()

plt.tight_layout()
plt.show()

print("\n观察: 大部分权重集中在零附近，这些小权重可能可以被剪枝!")

## 2. 非结构化剪枝

**核心概念**: 非结构化剪枝移除单个权重，产生稀疏矩阵

```
┌─────────────────────────────────────────────────────────────┐
│                   非结构化剪枝流程                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 计算每个权重的重要性                                    │
│     ┌─────────────────────────────────────────────────┐    │
│     │  importance = |weight|  (幅度剪枝)              │    │
│     └─────────────────────────────────────────────────┘    │
│                        ↓                                    │
│  2. 根据稀疏度确定阈值                                      │
│     ┌─────────────────────────────────────────────────┐    │
│     │  threshold = quantile(importance, sparsity)     │    │
│     └─────────────────────────────────────────────────┘    │
│                        ↓                                    │
│  3. 创建掩码，将小于阈值的权重置零                          │
│     ┌─────────────────────────────────────────────────┐    │
│     │  mask = importance > threshold                  │    │
│     │  weight = weight * mask                         │    │
│     └─────────────────────────────────────────────────┘    │
│                                                             │
│  优点: 高压缩率 (可达 90%+)                                 │
│  缺点: 需要特殊硬件支持才能加速                             │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 非结构化剪枝实现
# ============================================================

def magnitude_pruning(weight, sparsity):
    """
    基于幅度的非结构化剪枝
    
    参数:
        weight: 权重张量
        sparsity: 剪枝比例 (0-1)，如 0.5 表示剪掉 50%
        
    返回:
        mask: 保留的权重掩码 (1=保留, 0=剪枝)
    """
    # 计算重要性 (绝对值)
    importance = weight.abs().flatten()
    
    # 计算阈值: 找到第 sparsity 百分位的值
    threshold = torch.quantile(importance, sparsity)
    
    # 创建掩码: 大于阈值的保留
    mask = (weight.abs() > threshold).float()
    
    return mask


def compute_sparsity(tensor):
    """计算张量的稀疏度 (零元素比例)"""
    return (tensor == 0).float().mean().item()


print("非结构化剪枝函数定义完成!")

In [ ]:
# ============================================================
# 使用 PyTorch 内置剪枝 API
# ============================================================
print("=" * 60)
print("非结构化剪枝演示 (PyTorch API)")
print("=" * 60)

# 创建新模型
model_unstructured = ConvNet()

# 剪枝前的稀疏度
print(f"\n剪枝前:")
print(f"  conv1 稀疏度: {compute_sparsity(model_unstructured.conv1.weight):.2%}")
print(f"  fc1 稀疏度: {compute_sparsity(model_unstructured.fc1.weight):.2%}")

# 应用非结构化剪枝 (L1 范数，剪枝 50%)
# prune.l1_unstructured: 基于 L1 范数的非结构化剪枝
# amount=0.5 表示剪掉 50% 的权重
prune.l1_unstructured(model_unstructured.conv1, name='weight', amount=0.5)
prune.l1_unstructured(model_unstructured.conv2, name='weight', amount=0.5)
prune.l1_unstructured(model_unstructured.fc1, name='weight', amount=0.5)
prune.l1_unstructured(model_unstructured.fc2, name='weight', amount=0.5)

# 剪枝后的稀疏度
print(f"\n剪枝后 (50% 稀疏度):")
print(f"  conv1 稀疏度: {compute_sparsity(model_unstructured.conv1.weight):.2%}")
print(f"  conv2 稀疏度: {compute_sparsity(model_unstructured.conv2.weight):.2%}")
print(f"  fc1 稀疏度: {compute_sparsity(model_unstructured.fc1.weight):.2%}")
print(f"  fc2 稀疏度: {compute_sparsity(model_unstructured.fc2.weight):.2%}")

# 查看剪枝掩码
print(f"\n剪枝机制说明:")
print(f"  PyTorch 剪枝通过 mask 实现，不是真正删除权重")
print(f"  conv1 有 weight_mask 属性: {hasattr(model_unstructured.conv1, 'weight_mask')}")

In [ ]:
# ============================================================
# 可视化剪枝效果
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 原始权重
original_weight = model.conv1.weight.detach()[0, 0].numpy()
im1 = axes[0].imshow(original_weight, cmap='RdBu', vmin=-0.5, vmax=0.5)
axes[0].set_title('原始 Conv1 权重 (第一个滤波器)', fontsize=12)
plt.colorbar(im1, ax=axes[0])

# 剪枝后权重
pruned_weight = model_unstructured.conv1.weight.detach()[0, 0].numpy()
im2 = axes[1].imshow(pruned_weight, cmap='RdBu', vmin=-0.5, vmax=0.5)
axes[1].set_title('剪枝后 Conv1 权重 (50% 稀疏)', fontsize=12)
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

print("\n观察: 剪枝后的权重中，小幅度的值被置为零 (白色区域)")

## 3. 结构化剪枝

**核心概念**: 结构化剪枝移除整个结构单元（通道、滤波器、注意力头），产生规则的小模型

```
┌─────────────────────────────────────────────────────────────┐
│                   结构化剪枝类型                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 通道剪枝 (Channel Pruning)                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Conv 层: [out_ch, in_ch, H, W]                     │   │
│  │                                                     │   │
│  │  原始: 64 个输出通道                                │   │
│  │  剪枝: 移除 16 个不重要的通道                       │   │
│  │  结果: 48 个输出通道 (真正的小模型)                 │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  2. 滤波器剪枝 (Filter Pruning)                             │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  移除整个 3x3 滤波器                                │   │
│  │  等价于移除输出通道                                 │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  3. 注意力头剪枝 (Attention Head Pruning)                   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Transformer: 12 个注意力头                         │   │
│  │  剪枝: 移除 4 个不重要的头                          │   │
│  │  结果: 8 个注意力头                                 │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  优点: 无需特殊硬件，直接获得加速                           │
│  缺点: 压缩率通常低于非结构化剪枝                           │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 结构化剪枝实现
# ============================================================
print("=" * 60)
print("结构化剪枝演示")
print("=" * 60)

# 创建新模型
model_structured = ConvNet()

print(f"\n剪枝前:")
print(f"  conv1 权重形状: {model_structured.conv1.weight.shape}")
print(f"  conv2 权重形状: {model_structured.conv2.weight.shape}")

# ============================================================
# 结构化剪枝: 移除整个输出通道
# ============================================================
# prune.ln_structured: 基于 Ln 范数的结构化剪枝
# n=2 表示使用 L2 范数
# dim=0 表示沿输出通道维度剪枝
prune.ln_structured(model_structured.conv1, name='weight', amount=0.25, n=2, dim=0)
prune.ln_structured(model_structured.conv2, name='weight', amount=0.25, n=2, dim=0)

print(f"\n剪枝后 (25% 通道被置零):")
print(f"  conv1 权重形状: {model_structured.conv1.weight.shape}")
print(f"  conv2 权重形状: {model_structured.conv2.weight.shape}")

# 统计被剪枝的通道数
conv1_pruned_channels = (model_structured.conv1.weight.sum(dim=(1,2,3)) == 0).sum().item()
conv2_pruned_channels = (model_structured.conv2.weight.sum(dim=(1,2,3)) == 0).sum().item()

print(f"\n被剪枝的通道数:")
print(f"  conv1: {conv1_pruned_channels}/32 通道被置零")
print(f"  conv2: {conv2_pruned_channels}/64 通道被置零")

print(f"\n注意: 结构化剪枝后，权重形状不变，但整个通道被置零")
print(f"      需要额外步骤才能真正移除这些通道，获得小模型")

## 4. 重要性评估方法

**核心概念**: 如何判断哪些参数/通道是"不重要"的？

```
┌─────────────────────────────────────────────────────────────┐
│                   重要性评估方法对比                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 幅度剪枝 (Magnitude Pruning)                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  importance = |weight|                              │   │
│  │                                                     │   │
│  │  假设: 权重绝对值越小，对输出贡献越小               │   │
│  │  优点: 简单高效，无需额外计算                       │   │
│  │  缺点: 可能不准确，小权重可能很重要                 │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  2. 梯度剪枝 (Gradient-based Pruning)                       │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  importance = |weight × gradient|                   │   │
│  │                                                     │   │
│  │  假设: 权重和梯度都大，说明对损失影响大             │   │
│  │  优点: 考虑了训练动态                               │   │
│  │  缺点: 需要额外的反向传播                           │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  3. Taylor 展开剪枝                                         │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  ΔL ≈ g × δw + ½ h × δw²                           │   │
│  │  importance = |g × w| (一阶近似)                    │   │
│  │                                                     │   │
│  │  假设: 移除参数对损失的影响可用 Taylor 展开近似     │   │
│  │  优点: 理论基础强                                   │   │
│  │  缺点: 计算开销大                                   │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 重要性评估方法实现
# ============================================================

def magnitude_importance(weight):
    """
    幅度重要性: importance = |weight|
    
    最简单的方法，假设小权重不重要
    """
    return weight.abs()


def gradient_importance(weight, gradient):
    """
    梯度重要性: importance = |weight × gradient|
    
    结合权重和梯度信息
    """
    return (weight * gradient).abs()


def taylor_importance(weight, gradient):
    """
    Taylor 一阶重要性: importance = |gradient × weight|
    
    基于移除参数对损失的影响
    """
    return (gradient * weight).abs()


# ============================================================
# 对比不同重要性评估方法
# ============================================================
print("=" * 60)
print("重要性评估方法对比")
print("=" * 60)

# 创建模型并模拟一次前向-反向传播
model_importance = ConvNet()
dummy_input = torch.randn(1, 1, 28, 28)
dummy_target = torch.tensor([5])

# 前向传播
output = model_importance(dummy_input)
loss = F.cross_entropy(output, dummy_target)

# 反向传播获取梯度
loss.backward()

# 获取 conv1 的权重和梯度
weight = model_importance.conv1.weight.data
gradient = model_importance.conv1.weight.grad

# 计算不同方法的重要性
mag_imp = magnitude_importance(weight)
grad_imp = gradient_importance(weight, gradient)
taylor_imp = taylor_importance(weight, gradient)

print(f"\nConv1 层重要性统计:")
print(f"  幅度重要性:   均值={mag_imp.mean():.4f}, 最大={mag_imp.max():.4f}")
print(f"  梯度重要性:   均值={grad_imp.mean():.6f}, 最大={grad_imp.max():.6f}")
print(f"  Taylor重要性: 均值={taylor_imp.mean():.6f}, 最大={taylor_imp.max():.6f}")

## 5. 迭代剪枝

**核心概念**: 一次性剪枝可能导致较大精度损失，迭代剪枝逐步移除参数并微调

```
┌─────────────────────────────────────────────────────────────┐
│                   迭代剪枝流程                               │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  for iteration in range(num_iterations):                    │
│      1. 评估参数重要性                                      │
│      2. 剪枝 p% 的参数                                      │
│      3. 微调模型恢复精度 (几个 epoch)                       │
│      4. 评估模型性能                                        │
│                                                             │
│  最终稀疏度计算:                                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  最终稀疏度 = 1 - (1-p)^num_iterations              │   │
│  │                                                     │   │
│  │  例: p=20%, iterations=5                            │   │
│  │      最终稀疏度 = 1 - 0.8^5 ≈ 67%                   │   │
│  │                                                     │   │
│  │  例: p=20%, iterations=10                           │   │
│  │      最终稀疏度 = 1 - 0.8^10 ≈ 89%                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  优点: 精度损失小，可达到更高稀疏度                         │
│  缺点: 需要多次训练，时间成本高                            │
│                                                             │
└─────────────────────────────────────────────────────────────┘

一次性剪枝 vs 迭代剪枝:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  精度                                                       │
│   ↑                                                         │
│   │  ●────────────────────● 迭代剪枝 (精度保持好)          │
│   │       ╲                                                 │
│   │        ╲                                                │
│   │         ●──────────────● 一次性剪枝 (精度下降快)       │
│   │                                                         │
│   └──────────────────────────────→ 稀疏度                   │
│        20%    40%    60%    80%                             │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 迭代剪枝实现
# ============================================================

def iterative_pruning(model, prune_ratio_per_iter=0.2, num_iterations=5):
    """
    迭代剪枝
    
    参数:
        model: 待剪枝模型
        prune_ratio_per_iter: 每次迭代剪枝比例
        num_iterations: 迭代次数
        
    返回:
        sparsity_history: 每次迭代后的稀疏度
    """
    sparsity_history = []
    
    for i in range(num_iterations):
        # 对每个需要剪枝的层应用剪枝
        for name, module in model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                # 跳过已经剪枝过的层（检查是否有 weight_orig）
                if hasattr(module, 'weight_orig'):
                    # 已经有剪枝，继续剪枝
                    prune.l1_unstructured(module, name='weight', amount=prune_ratio_per_iter)
                else:
                    # 首次剪枝
                    prune.l1_unstructured(module, name='weight', amount=prune_ratio_per_iter)
        
        # 计算当前稀疏度
        total_zeros = 0
        total_params = 0
        for name, module in model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                total_zeros += (module.weight == 0).sum().item()
                total_params += module.weight.numel()
        
        current_sparsity = total_zeros / total_params
        sparsity_history.append(current_sparsity)
        
        # 这里应该进行微调 (fine-tuning)
        # 为简化演示，我们跳过微调步骤
    
    return sparsity_history


# ============================================================
# 迭代剪枝演示
# ============================================================
print("=" * 60)
print("迭代剪枝演示")
print("=" * 60)

# 创建新模型
model_iterative = ConvNet()

# 执行迭代剪枝
sparsity_history = iterative_pruning(
    model_iterative, 
    prune_ratio_per_iter=0.2,  # 每次剪枝 20%
    num_iterations=5           # 迭代 5 次
)

print(f"\n迭代剪枝结果:")
for i, sparsity in enumerate(sparsity_history):
    theoretical = 1 - (1 - 0.2) ** (i + 1)
    print(f"  迭代 {i+1}: 稀疏度 = {sparsity:.2%} (理论值: {theoretical:.2%})")

print(f"\n最终稀疏度: {sparsity_history[-1]:.2%}")
print(f"理论最终稀疏度: {1 - 0.8**5:.2%}")

In [ ]:
# ============================================================
# 可视化迭代剪枝过程
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 图1: 稀疏度随迭代增长
iterations = range(1, len(sparsity_history) + 1)
theoretical_sparsity = [1 - 0.8**i for i in iterations]

axes[0].plot(iterations, sparsity_history, 'bo-', label='实际稀疏度', linewidth=2, markersize=8)
axes[0].plot(iterations, theoretical_sparsity, 'r--', label='理论稀疏度', linewidth=2)
axes[0].set_xlabel('迭代次数')
axes[0].set_ylabel('稀疏度')
axes[0].set_title('迭代剪枝: 稀疏度增长曲线', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 图2: 不同剪枝策略对比
strategies = ['一次性 50%', '一次性 70%', '迭代 5次\n(每次20%)', '迭代 10次\n(每次10%)']
sparsities = [0.5, 0.7, 1-0.8**5, 1-0.9**10]
colors = ['blue', 'blue', 'green', 'green']

axes[1].bar(strategies, sparsities, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('最终稀疏度')
axes[1].set_title('不同剪枝策略的最终稀疏度', fontsize=12)
for i, v in enumerate(sparsities):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\n结论: 迭代剪枝可以达到更高的稀疏度，同时保持更好的精度")

## 6. 总结

### 剪枝方法对比

| 方法 | 压缩率 | 加速效果 | 实现难度 | 硬件要求 | 适用场景 |
|:-----|:------:|:--------:|:--------:|:--------:|:---------|
| 非结构化剪枝 | 高 (90%+) | 需要稀疏计算支持 | ⭐ | 特殊硬件 | 稀疏计算硬件 |
| 结构化剪枝 | 中 (50-70%) | 直接加速 | ⭐⭐ | 通用硬件 | CNN、Transformer |
| 迭代剪枝 | 高 | 取决于剪枝类型 | ⭐⭐⭐ | 取决于类型 | 高压缩率需求 |

### 重要性评估方法对比

| 方法 | 公式 | 优点 | 缺点 |
|:-----|:-----|:-----|:-----|
| 幅度剪枝 | `|w|` | 简单高效 | 可能不准确 |
| 梯度剪枝 | `|w × ∇w|` | 考虑训练动态 | 需要额外计算 |
| Taylor 展开 | `|g × w|` | 理论基础强 | 计算开销大 |

### 最佳实践

```
剪枝检查清单:
✓ 选择合适的剪枝类型 (结构化 vs 非结构化)
✓ 从小稀疏度开始，逐步增加
✓ 剪枝后进行微调恢复精度
✓ 在目标硬件上验证加速效果
✓ 监控精度变化，设置可接受阈值

常见陷阱:
✗ 一次性剪枝过多导致精度崩溃
✗ 忽略剪枝后的微调步骤
✗ 非结构化剪枝期望在通用硬件上加速
✗ 剪枝敏感层 (如第一层、最后一层)
```

### PyTorch 剪枝 API 速查

```python
import torch.nn.utils.prune as prune

# 非结构化剪枝
prune.l1_unstructured(module, 'weight', amount=0.3)      # L1 范数
prune.random_unstructured(module, 'weight', amount=0.3)  # 随机

# 结构化剪枝
prune.ln_structured(module, 'weight', amount=0.3, n=2, dim=0)  # Ln 范数

# 全局剪枝 (跨层统一阈值)
prune.global_unstructured(parameters, pruning_method=prune.L1Unstructured, amount=0.3)

# 移除剪枝重参数化 (使剪枝永久化)
prune.remove(module, 'weight')
```

### 下一步学习

- **03_Distillation_tutorial.ipynb**: 知识蒸馏
- **04_Export_tutorial.ipynb**: 模型导出与部署
- **05_Advanced_Optimization_tutorial.ipynb**: 高级优化技术

In [ ]:
# ============================================================
# 可视化不同重要性评估方法
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 计算每个通道的重要性 (用于结构化剪枝)
channel_mag_imp = mag_imp.sum(dim=(1, 2, 3)).numpy()
channel_grad_imp = grad_imp.sum(dim=(1, 2, 3)).numpy()
channel_taylor_imp = taylor_imp.sum(dim=(1, 2, 3)).numpy()

# 绘制柱状图
axes[0].bar(range(32), channel_mag_imp, color='blue', alpha=0.7)
axes[0].set_title('幅度重要性 (按通道)', fontsize=11)
axes[0].set_xlabel('通道索引')
axes[0].set_ylabel('重要性')

axes[1].bar(range(32), channel_grad_imp, color='green', alpha=0.7)
axes[1].set_title('梯度重要性 (按通道)', fontsize=11)
axes[1].set_xlabel('通道索引')
axes[1].set_ylabel('重要性')

axes[2].bar(range(32), channel_taylor_imp, color='red', alpha=0.7)
axes[2].set_title('Taylor重要性 (按通道)', fontsize=11)
axes[2].set_xlabel('通道索引')
axes[2].set_ylabel('重要性')

plt.tight_layout()
plt.show()

print("\n观察: 不同方法可能给出不同的重要性排序")
print("      实践中，幅度剪枝最常用，因为简单且效果不错")